# Moral foundation coding with an OpenAI model — main analysis

The baseline cell: **main prompt × 0-shot**, model `gpt-6-luna` (non-reasoning, `reasoning_effort="none"`).

Before running:
* Select the Python kernel (top right) you want to use.
* Provide your OpenAI API key (prompted below, or set `OPENAI_API_KEY` in a `.env` file — keep `.env` out of git).

## Setup

In [ ]:
# Run once, then restart the kernel
%pip install -q -U openai pandas tqdm scikit-learn ipywidgets python-dotenv

In [36]:
# Reload run_grid_openai.py automatically whenever you edit it
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [37]:
import os, json, getpass
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support, cohen_kappa_score

try:
    from dotenv import load_dotenv
    load_dotenv()                      # reads OPENAI_API_KEY from .env if present
except ImportError:
    pass
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

import run_grid_openai as rg
print("model:", rg.MODEL, "| reasoning_effort:", rg.REASONING_EFFORT, "| output folder:", rg.OUT_DIR)

model: gpt-6-luna | reasoning_effort: none | output folder: runs/gpt-6-luna


## Load validation set

In [38]:
validation = rg.load_validation()
print(validation.shape)
validation.head()

(50, 20)


,para_id,moral_Yunfan,care_Yunfan,fairness_Yunfan,loyalty_Yunfan,authority_Yunfan,sanctity_Yunfan,moral_Wiktoria,care_Wiktoria,fairness_Wiktoria,loyalty_Wiktoria,authority_Wiktoria,sanctity_Wiktoria,moral_Carlos,care_Carlos,fairness_Carlos,loyalty_Carlos,authority_Carlos,sanctity_Carlos,text
0,CREC-2025-10-21-pt2-PgS7587-2_6_214,1,0,0,0,1,0,1,0,0,0,1,1,1,0,0,0,1,0,"Well, this has been quite a list of assaults b..."
1,CREC-2025-11-18-pt1-PgH4718-6_32_0,1,0,1,0,1,1,1,0,1,0,1,1,1,0,0,0,0,1,It is time. They are going to have a reckoning...
2,CREC-2025-12-18-pt1-PgH6099-7_10_9,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,When the United States hesitates or sends mixe...
3,CREC-2026-01-06-pt1-PgS53_0_16,1,0,0,1,1,1,1,0,0,1,0,1,1,0,0,0,1,0,I find myself thinking about those foundationa...
4,CREC-2025-09-03-pt1-PgS6006_0_6,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,The assistance and know-how the team at Fronti...


## Prompt
The main prompt is identical to the one in `llm_moral_coding.ipynb`.

In [39]:
print(rg.make_prompt(validation["text"][0]))



1. TASK
You're an expert political scientist analyzing U.S. congressional floor speeches. You need to decide whether a paragraph from the speeches uses moral rhetoric, and if it does, which moral foundations it draws on.
Code six labels, each as 1 (present) or 0 (absent): MORAL RHETORIC, CARE, FAIRNESS, LOYALTY, AUTHORITY, SANCTITY.

2. CODING APPROACH
* The unit of analysis is the paragraph. Read the full paragraph before coding.
* Code each label for presence (1) or absence (0).
* Code a label as 1 if the paragraph expresses its meaning at least once.
* A paragraph can be coded for several moral foundations at the same time.

Procedure:
1. Read the full paragraph.
2. Code MORAL RHETORIC as 1 or 0.
3. If MORAL RHETORIC is 1, code each of the five moral foundations as 1 or 0. At least one foundation must be 1.
4. If MORAL RHETORIC is 0, all five moral foundations must be 0.

3. LABELS


Label: MORAL RHETORIC
Definition: The speaker, explicitly or implicitly,
* judges an action, actor

Test on one paragraph:

In [40]:
rg.check_params(rg.make_prompt(validation["text"][0]))   # drops temperature/seed if the model rejects them
rg.ask_llm(rg.make_prompt(validation["text"][0]), validation["text"][0])

{'moral_rhetoric': 1,
 'care': 0,
 'fairness': 0,
 'loyalty': 0,
 'authority': 1,
 'sanctity': 0,
 'evidence_moral_rhetoric': 'assaults by this government on freedom of the press',
 'evidence_care': '',
 'evidence_fairness': '',
 'evidence_loyalty': '',
 'evidence_authority': 'assaults by this government on freedom of the press',
 'evidence_sanctity': '',
 'problem': None,
 'warnings': [],
 'raw': '{"evidence":{"moral_rhetoric":"assaults by this government on freedom of the press","care":"","fairness":"","loyalty":"","authority":"assaults by this government on freedom of the press","sanctity":""},"moral_rhetoric":1,"care":0,"fairness":0,"loyalty":0,"authority":1,"sanctity":0}',
 'model_version': 'gpt-6-luna'}

## Run the primary cell (main × 0-shot)

Every paragraph is coded once; no resume logic. Results are saved to
`runs/gpt-6-luna/main_0shot_validation.csv`, together with the exact prompt (`_prompt.txt`)
and the settings used (`_settings.json`).

In [41]:
pred = rg.run_cell(validation, "main", 0)
pred.head()

main, 0-shot:   0%|          | 0/50 [00:00<?, ?it/s]

,para_id,text,variant,shots,moral_rhetoric,care,fairness,loyalty,authority,sanctity,evidence_moral_rhetoric,evidence_care,evidence_fairness,evidence_loyalty,evidence_authority,evidence_sanctity,problem,warnings,raw,model_version
0,CREC-2025-10-21-pt2-PgS7587-2_6_214,"Well, this has been quite a list of assaults b...",main,0,1,0,0,0,1,0,assaults by this government on freedom of the ...,,,,assaults by this government on freedom of the ...,,None,[],"{""evidence"":{""moral_rhetoric"":""assaults by thi...",gpt-6-luna
1,CREC-2025-11-18-pt1-PgH4718-6_32_0,It is time. They are going to have a reckoning...,main,0,1,0,0,1,0,1,“rotten system”,,,“everyone in between is finally waking up agai...,,“rotten system”,None,[],"{""evidence"":{""moral_rhetoric"":""“rotten system”...",gpt-6-luna
2,CREC-2025-12-18-pt1-PgH6099-7_10_9,When the United States hesitates or sends mixe...,main,0,0,0,0,0,0,0,,,,,,,None,[],"{""evidence"":{""moral_rhetoric"":"""",""care"":"""",""fa...",gpt-6-luna
3,CREC-2026-01-06-pt1-PgS53_0_16,I find myself thinking about those foundationa...,main,0,1,0,0,1,1,1,“secure the Blessings of Liberty to ourselves ...,,,“to ourselves and our Posterity”,“a specific phrase from the preamble”,“those foundational values”,None,[],"{""evidence"":{""moral_rhetoric"":""“secure the Ble...",gpt-6-luna
4,CREC-2025-09-03-pt1-PgS6006_0_6,The assistance and know-how the team at Fronti...,main,0,0,0,0,0,0,0,,,,,,,None,[],"{""evidence"":{""moral_rhetoric"":"""",""care"":"""",""fa...",gpt-6-luna


In [42]:
settings = json.load(open(f"{rg.OUT_DIR}/main_0shot_validation_settings.json"))
print("dropped parameters:", settings["dropped_params"])     # [] = temperature and seed were used
print("model versions returned:", settings["model_versions"])
print("params:", settings["params"])

dropped parameters: []
model versions returned: ['gpt-6-luna']
params: {'model': 'gpt-6-luna', 'temperature': 0, 'seed': 502, 'reasoning_effort': 'none'}


## Output checks

In [43]:
print("problems:")
print(pred["problem"].fillna("none").value_counts())
warnings = pred["warnings"].apply(lambda w: w if isinstance(w, list) else [])
print("\nwarnings:")
print(pd.Series([x.split(": ", 1)[1] for ws in warnings for x in ws], dtype=str).value_counts())

problems:
problem
none                        49
moral_without_foundation     1
Name: count, dtype: int64

warnings:
evidence not found in paragraph    6
evidence given but coded 0         2
Name: count, dtype: int64


## Evaluation against human labels

In [44]:
EVAL_LABELS = ["moral"] + rg.FOUNDATIONS
CODERS = ["Yunfan", "Wiktoria", "Carlos"]
LLM_COL = {"moral": "moral_rhetoric", **{f: f for f in rg.FOUNDATIONS}}

m = validation.merge(pred.drop(columns=["text"], errors="ignore"), on="para_id")
m = m[m["moral_rhetoric"].notna()]          # drop parse/API errors
for l in EVAL_LABELS:                        # majority vote of the three coders
    m[f"{l}_majority"] = (m[[f"{l}_{c}" for c in CODERS]].sum(axis=1) >= 2).astype(int)

def evaluate(df, human_col_fn, llm_col_map=LLM_COL):
    rows = []
    for l in EVAL_LABELS:
        sub = df[[human_col_fn(l), llm_col_map[l]]].dropna()
        y, yhat = sub.iloc[:, 0].astype(int), sub.iloc[:, 1].astype(int)
        p, r, f, _ = precision_recall_fscore_support(y, yhat, average="binary", zero_division=0)
        rows.append({"label": l, "n": len(sub), "human_n": y.sum(), "llm_n": yhat.sum(),
                     "accuracy": (y == yhat).mean(), "precision": p, "recall": r, "f1": f,
                     "kappa": cohen_kappa_score(y, yhat)})
    return pd.DataFrame(rows).set_index("label").round(3)

all_results = {c: evaluate(m, lambda l, c=c: f"{l}_{c}") for c in CODERS}
all_results["majority"] = evaluate(m, lambda l: f"{l}_majority")
results = pd.concat(all_results, names=["coder", "label"])

print("n =", len(m))
for c, res in all_results.items():
    print(f"{c:>9} macro-F1 (5 foundations):", res.loc[rg.FOUNDATIONS, "f1"].mean().round(3))
results

n = 50
   Yunfan macro-F1 (5 foundations): 0.644
 Wiktoria macro-F1 (5 foundations): 0.622
   Carlos macro-F1 (5 foundations): 0.757
 majority macro-F1 (5 foundations): 0.684


n  human_n  llm_n  accuracy  precision  recall     f1  \
coder    label                                                               
Yunfan   moral      50       28     25      0.78      0.840   0.750  0.792   
         care       50       13     10      0.86      0.800   0.615  0.696   
         fairness   50       12      9      0.86      0.778   0.583  0.667   
         loyalty    50        4      4      0.92      0.500   0.500  0.500   
         authority  50       13     13      0.84      0.692   0.692  0.692   
         sanctity   50        5      4      0.94      0.750   0.600  0.667   
Wiktoria moral      50       30     25      0.78      0.880   0.733  0.800   
         care       50       15     10      0.86      0.900   0.600  0.720   
         fairness   50       13      9      0.88      0.889   0.615  0.727   
         loyalty    50        5      4      0.90      0.500   0.400  0.444   
         authority  50       16     13      0.78      0.692   0.562  0.621   
         sanctity   50        6      4      0.92      0.750   0.500  0.600   
Carlos   moral      50       27     25      0.92      0.960   0.889  0.923   
         care       50       12     10      0.92      0.900   0.750  0.818   
         fairness   50        6      9      0.94      0.667   1.000  0.800   
         loyalty    50        2      4      0.96      0.500   1.000  0.667   
         authority  50       11     13      0.92      0.769   0.909  0.833   
         sanctity   50        2      4      0.96      0.500   1.000  0.667   
majority moral      50       29     25      0.80      0.880   0.759  0.815   
         care       50       14     10      0.88      0.900   0.643  0.750   
         fairness   50       13      9      0.88      0.889   0.615  0.727   
         loyalty    50        4      4      0.92      0.500   0.500  0.500   
         authority  50       13     13      0.84      0.692   0.692  0.692   
         sanctity   50        4      4      0.96      0.750   0.750  0.750   

                    kappa  
coder    label             
Yunfan   moral      0.560  
         care       0.607  
         fairness   0.580  
         loyalty    0.457  
         authority  0.584  
         sanctity   0.634  
Wiktoria moral      0.560  
         care       0.632  
         fairness   0.654  
         loyalty    0.390  
         authority  0.468  
         sanctity   0.558  
Carlos   moral      0.840  
         care       0.767  
         fairness   0.766  
         loyalty    0.648  
         authority  0.781  
         sanctity   0.648  
majority moral      0.600  
         care       0.674  
         fairness   0.654  
         loyalty    0.457  
         authority  0.584  
         sanctity   0.728

In [45]:
results.to_csv(f"{rg.OUT_DIR}/main_0shot_evaluation.csv")

### LLM–human agreement vs. human–human agreement (Cohen's κ)

Is the LLM within the range of agreement among the human coders?

In [46]:
from itertools import combinations

rows = []
for l in EVAL_LABELS:
    hh = [cohen_kappa_score(m[f"{l}_{a}"].astype(int), m[f"{l}_{b}"].astype(int))
          for a, b in combinations(CODERS, 2)]
    lh = [cohen_kappa_score(m[f"{l}_{c}"].astype(int), m[LLM_COL[l]].astype(int)) for c in CODERS]
    rows.append({"label": l,
                 "human-human mean": sum(hh) / 3, "human-human min": min(hh), "human-human max": max(hh),
                 "LLM-human mean": sum(lh) / 3, "LLM-human min": min(lh), "LLM-human max": max(lh)})
pd.DataFrame(rows).set_index("label").round(3)

,human-human mean,human-human min,human-human max,LLM-human mean,LLM-human min,LLM-human max
label,,,,,,
moral,0.784,0.715,0.918,0.653,0.560,0.840
care,0.794,0.733,0.901,0.669,0.607,0.767
fairness,0.659,0.471,0.947,0.667,0.580,0.766
loyalty,0.472,0.242,0.878,0.498,0.390,0.648
authority,0.696,0.549,0.781,0.611,0.468,0.781
sanctity,0.569,0.468,0.694,0.613,0.558,0.648


## Disagreements for error analysis

In [47]:
EVID_COL = {l: f"evidence_{LLM_COL[l]}" for l in EVAL_LABELS}

def find_disagreements(df, coder="majority", outfile=None):
    """coder: 'Yunfan' / 'Wiktoria' / 'Carlos' / 'majority'"""
    def diff(row):
        out = []
        for l in EVAL_LABELS:
            h, p = row[f"{l}_{coder}"], row[LLM_COL[l]]
            if pd.isna(h) or pd.isna(p):
                continue
            if int(h) != int(p):
                out.append(l)
        return out

    df = df.copy()
    df["disagree_on"] = df.apply(diff, axis=1)
    errors = df[df["disagree_on"].str.len() > 0]
    cols = (["para_id", "text", "disagree_on"]
            + [c for l in EVAL_LABELS for c in (f"{l}_{coder}", LLM_COL[l])]
            + [EVID_COL[l] for l in EVAL_LABELS])
    if outfile:
        errors[cols].to_csv(outfile, index=False)
    print(f"{coder}: {len(errors)} / {len(df)} paragraphs with at least one disagreement")
    return errors[cols]

errors = find_disagreements(m, coder="majority",
                            outfile=f"{rg.OUT_DIR}/main_0shot_disagreements_majority.csv")
errors

majority: 19 / 50 paragraphs with at least one disagreement


,para_id,text,disagree_on,moral_majority,moral_rhetoric,care_majority,care,fairness_majority,fairness,loyalty_majority,...,authority_majority,authority,sanctity_majority,sanctity,evidence_moral_rhetoric,evidence_care,evidence_fairness,evidence_loyalty,evidence_authority,evidence_sanctity
1,CREC-2025-11-18-pt1-PgH4718-6_32_0,It is time. They are going to have a reckoning...,"[fairness, loyalty, authority]",1,1,0,0,1,0,0,...,1,0,1,1,“rotten system”,,,“everyone in between is finally waking up agai...,,“rotten system”
5,CREC-2025-11-17-pt1-PgH4672-2_1_0,"Speaking of taxes, the tax on the economy and ...","[moral, loyalty]",1,0,0,0,0,0,1,...,0,0,0,0,,,,,,
6,CREC-2025-12-03-pt1-PgH5001-2_1_0,With only 10 voting days left in this year's c...,"[moral, care, fairness]",1,0,1,0,1,0,0,...,0,0,0,0,,"""avert a full-blown healthcare crisis""",,,,
10,CREC-2025-02-12-pt1-PgS955_0_11,The administration tried to freeze funding for...,"[moral, care]",1,0,1,0,0,0,0,...,0,0,0,0,,,,,,
11,CREC-2025-04-28-pt1-PgH1651_14_4,"However, we must not allow President Trump to ...",[authority],1,1,1,1,0,0,1,...,1,0,0,0,“careless policies”,“put this great progress in jeopardy”; “crash ...,,“stand with the domestic manufacturers they cl...,,
12,CREC-2025-05-14-pt1-PgS2920-5_9_6,I oppose all violence against all police offic...,[authority],1,1,1,1,1,1,0,...,0,1,0,0,I oppose all violence against all police offic...,violence against all police officers,why there seems to be an exemption for the vio...,,the officers who protect us,
13,CREC-2025-06-24-pt1-PgH2894_2_3,"Dave's deep knowledge, steady guidance, and pa...","[moral, care]",0,1,0,1,0,0,0,...,0,0,0,0,passion for serving others,serving others,,,,
14,CREC-2025-06-24-pt1-PgS3501_1_32,"Likewise, we can't expect Putin to end his agg...","[moral, loyalty, authority]",0,1,0,0,0,0,0,...,0,1,0,0,America's abandonment of Ukraine,,,America's abandonment of Ukraine,our own strength atrophy,
17,CREC-2026-01-13-pt1-PgH711_4_30,"Yes, once again, it awaits Senate action. The ...",[sanctity],1,1,0,0,0,0,0,...,1,1,1,0,“We must also defend the rule of law itself.”,,,,“We must also defend the rule of law itself.”,
18,CREC-2026-02-24-pt1-PgS638_0_4,"Now, I have served in the Capitol for over two...","[moral, loyalty]",1,0,0,0,0,0,1,...,0,0,0,0,,,,,,


# Full grid: 3 prompt variants × 3 shot conditions

Needs, in the same folder:
* `fewshot_examples.csv` — the few-shot examples in order (first 2 rows = 2-shot, first 5 = 5-shot).
* `DEFINITIONS_PARAPHRASE` in `run_grid_openai.py` — reviewed paraphrased definitions.

Example paragraphs that are also in the validation set are left out of the 2- and 5-shot cells,
so all cells are compared on the paragraphs they have in common.

In [48]:
examples = rg.load_examples()
examples[["para_id"] + rg.LABELS]
print(rg.make_examples(examples.head(2)))          # check the 2-shot block

Coded examples:

Example 1:
Paragraph: "For me, my pro-life stance is deeply rooted in my Christian faith and belief that all human life is precious, and it is a gift from the almighty God. Defending the unborn has long been one of my highest priorities, and the first term I got elected to Congress, one of my first votes was cast in favor of life."
Answer: {"evidence": {"moral_rhetoric": "all human life is precious", "care": "Defending the unborn has long been one of my highest priorities", "fairness": "", "loyalty": "", "authority": "", "sanctity": "all human life is precious, and it is a gift from the almighty God"}, "moral_rhetoric": 1, "care": 1, "fairness": 0, "loyalty": 0, "authority": 0, "sanctity": 1}

Example 2:
Paragraph: "Every American should be outraged at how the Trump administration is treating anyone who dares to stand in their way. Whether you are a U.S. Senator, a U.S. Citizen, or an immigrant, no one is protected by the Trump regime. It is my job to ensure that my co

In [49]:
# The primary cell (main x 0-shot) was already run above as `pred`, so run the other 8 cells.
# Each is saved as runs/gpt-6-luna/{variant}_{shots}shot_validation.csv
other_cells = [(v, k) for v in rg.VARIANTS for k in rg.SHOTS if (v, k) != rg.PRIMARY]
all_pred = pd.concat([pred, rg.run_grid(validation, cells=other_cells, examples=examples)],
                     ignore_index=True)
all_pred.groupby(["variant", "shots"]).size()

main, 2-shot:   0%|          | 0/50 [00:00<?, ?it/s]

main, 5-shot:   0%|          | 0/50 [00:00<?, ?it/s]

paraphrase, 0-shot:   0%|          | 0/50 [00:00<?, ?it/s]

paraphrase, 2-shot:   0%|          | 0/50 [00:00<?, ?it/s]

paraphrase, 5-shot:   0%|          | 0/50 [00:00<?, ?it/s]

reverse, 0-shot:   0%|          | 0/50 [00:00<?, ?it/s]

reverse, 2-shot:   0%|          | 0/50 [00:00<?, ?it/s]

reverse, 5-shot:   0%|          | 0/50 [00:00<?, ?it/s]

variant     shots
main        0        50
            2        50
            5        50
paraphrase  0        50
            2        50
            5        50
reverse     0        50
            2        50
            5        50
dtype: int64

In [50]:
# Evaluate each model condition against human benchmark and save the results to runs/gpt-6-luna/{variant}_{shots}shot_evaluation.csv
for (v, k), g in all_pred.groupby(["variant", "shots"]):
    mm = validation.merge(g.drop(columns=["text"], errors="ignore"), on="para_id")
    mm = mm[mm["moral_rhetoric"].notna()]
    for l in EVAL_LABELS:
        mm[f"{l}_majority"] = (mm[[f"{l}_{c}" for c in CODERS]].sum(axis=1) >= 2).astype(int)
    res = {c: evaluate(mm, lambda l, c=c: f"{l}_{c}") for c in CODERS}
    res["majority"] = evaluate(mm, lambda l: f"{l}_majority")
    pd.concat(res, names=["coder", "label"]).to_csv(f"{rg.OUT_DIR}/{v}_{k}shot_evaluation.csv")